# Dataset-agnostic pseudo-control pairing execution notebook

This notebook runs the standardized S0–S5 pseudo-control construction workflow for any processed Perturb-seq dataset with:

- one control AnnData file;
- one or more perturbed AnnData files, for example `single`, `dual`, or `multi` perturbations;
- optional control SEACell membership files, or the ability to run SEACells from the control h5ad.

The strategy names and order are fixed as:

1. `S0_naive_mean_control_reference`
2. `S1_random_single_control`
3. `S2_random_average_controls`
4. `S3_SEACell_metacell_average`
5. `S4_SEACell_balanced_random_sample`
6. `S5_SEACell_OT_sampled_average`


In [1]:
from pathlib import Path
import sys

# If this notebook is copied together with the .py files, set WORKFLOW_DIR to that folder.
# On IBEX, I recommend copying the whole folder to something like:
# /ibex/user/chenj0i/Perturbation/scripts/pseudo_pairing_refactor
WORKFLOW_DIR = Path.cwd()

# Example explicit path if you copy the folder to your scripts directory:
# WORKFLOW_DIR = Path('/ibex/user/chenj0i/Perturbation/scripts/pseudo_pairing_refactor')

sys.path.insert(0, str(WORKFLOW_DIR))
print('Workflow dir:', WORKFLOW_DIR)


Workflow dir: /ibex/user/chenj0i/Perturbation/SEACells/Junfan_scripts/Replogle_RPE/pseudo_pairing_refactor


In [2]:
from run_pseudo_pairing_repetitions import run_pseudo_pairing_repetition_plan
from pseudo_pairing_utils import STRATEGY_ORDER

print('Available strategy order:')
for s in STRATEGY_ORDER:
    print(' -', s)


Available strategy order:
 - S0_naive_mean_control_reference
 - S1_random_single_control
 - S2_random_average_controls
 - S3_SEACell_metacell_average
 - S4_SEACell_balanced_random_sample
 - S5_SEACell_OT_sampled_average


## 1. Configure dataset paths

The paths below are written for the Norman dataset generated by the preprocessing notebook.  For Replogle or another dataset, only change `DATASET_ID`, `CONTROL_H5AD`, `PERTURBED_H5ADS`, and `OUTDIR`.

The workflow accepts multiple perturbed datasets at once.  Keep only the keys you want to run, for example only `single`, or `single + dual + multi`.


In [3]:
# ============================================================
# Dataset paths
# ============================================================
DATASET_ID = 'Replogle_RPE'

PROCESSED_ROOT = Path('/ibex/user/chenj0i/Perturbation/data/processed_data') / DATASET_ID
GROUP_ROOT = PROCESSED_ROOT / 'groups'

CONTROL_H5AD = GROUP_ROOT / f'{DATASET_ID}_control_processed.h5ad'

# Keep or remove groups depending on which files exist after preprocessing.
PERTURBED_H5ADS = {
    'single': GROUP_ROOT / f'{DATASET_ID}_single_processed.h5ad',
    # 'dual': GROUP_ROOT / f'{DATASET_ID}_dual_processed.h5ad',
    # 'multi': GROUP_ROOT / f'{DATASET_ID}_multi_processed.h5ad',
}

OUTDIR = Path('/ibex/project/c2366/Perturb_data/Replogle_rpe_data')

print('Control:', CONTROL_H5AD)
print('Perturbed groups:')
for k, v in PERTURBED_H5ADS.items():
    print(f'  {k}: {v}')
print('Output root:', OUTDIR / DATASET_ID)


Control: /ibex/user/chenj0i/Perturbation/data/processed_data/Replogle_RPE/groups/Replogle_RPE_control_processed.h5ad
Perturbed groups:
  single: /ibex/user/chenj0i/Perturbation/data/processed_data/Replogle_RPE/groups/Replogle_RPE_single_processed.h5ad
Output root: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE


## 2. Define repeat grids and strategy parameters

For a quick smoke test, set `MAX_PAIRS_PER_PERTURBATION = 300`.  For the full dataset, keep it as `None`.

The SEACell membership root is separated from strategy outputs so S3, S4, and S5 reuse the same fixed control metacells.


In [4]:
# ============================================================
# Strategy selection
# ============================================================
STRATEGIES_TO_RUN = [
    'S0_naive_mean_control_reference',
    'S1_random_single_control',
    'S2_random_average_controls',
    'S3_SEACell_metacell_average',
    'S4_SEACell_balanced_random_sample',
    'S5_SEACell_OT_sampled_average',
]

# ============================================================
# Repetition settings
# ============================================================
RANDOM_SEEDS = [0, 1, 2, 3, 4]
# RANDOM_SEEDS = [4]

# Debug option. Use 300 for fast testing, None for the full dataset.
MAX_PAIRS_PER_PERTURBATION = None
PAIR_SELECTION_SEED = 42

# ============================================================
# Random baseline parameters
# ============================================================
S2_N_CONTROL_CELLS_TO_AVERAGE_VALUES = [100]
SAMPLING_REPLACE = True

# ============================================================
# SEACell settings reused by S3/S4/S5
# ============================================================
SEACELL_SETTINGS = [
    {
    'setting_id': 'nmc_50',
    'n_metacells': 50,
    'n_waypoint_eigs': 10,
    'seacells_n_iter': 70,
    'seacell_seed': 42,
    },
    {
        'setting_id': 'nmc_100',
        'n_metacells': 100,
        'n_waypoint_eigs': 10,
        'seacells_n_iter': 70,
        'seacell_seed': 42,
    },
    {
        'setting_id': 'nmc_200',
        'n_metacells': 200,
        'n_waypoint_eigs': 10,
        'seacells_n_iter': 70,
        'seacell_seed': 42,
    },
    {
        'setting_id': 'nmc_350',
        'n_metacells': 350,
        'n_waypoint_eigs': 10,
        'seacells_n_iter': 70,
        'seacell_seed': 42,
    },
    {
        'setting_id': 'nmc_500',
        'n_metacells': 500,
        'n_waypoint_eigs': 10,
        'seacells_n_iter': 70,
        'seacell_seed': 42,
    },
]

# S3: randomly sample k metacells and average their metacell expression profiles.
# S3_N_METACELLS_TO_AVERAGE_VALUES = [3]
S3_N_METACELLS_TO_AVERAGE_VALUES = [3, 5, 10]
SAMPLE_METACELLS_WITH_REPLACEMENT = True

# S5: OT top-k metacells, then sample true control cells from each matched metacell.
# S5_TOP_K_VALUES = [5]
S5_TOP_K_VALUES = [1, 3, 5]
SAMPLE_CELLS_PER_METACELL = 10
OT_REG = 0.05
CONTROL_MASS = 'size'  # 'size' or 'uniform'

# ============================================================
# Data keys
# ============================================================
EXPR_LAYER = 'X'
EMBEDDING_KEY = 'X_pca'

# Use 'auto' to infer from obs columns.  For your processed files, likely candidates are:
# Replogle: 'gene'
# Norman/scPerturb-style: 'condition' or the standardized 'perturbation_key'
PERTURBATION_KEY = 'auto'

## 3. Build the config object

The same config structure should work for Replogle, Norman, or later datasets after changing only paths and metadata keys.


In [5]:
CONFIG = {
    'dataset_id': DATASET_ID,
    'control_h5ad': str(CONTROL_H5AD),
    'perturbed_h5ads': {k: str(v) for k, v in PERTURBED_H5ADS.items()},
    'outdir': str(OUTDIR),

    # Which strategies to run.
    'strategies_to_run': STRATEGIES_TO_RUN,

    # Data keys.
    'expr_layer': EXPR_LAYER,
    'embedding_key': EMBEDDING_KEY,
    'perturbation_key': PERTURBATION_KEY,
    'require_all_genes': False,

    # Repeats and debug subset.
    'random_seeds': RANDOM_SEEDS,
    'max_pairs_per_perturbation': MAX_PAIRS_PER_PERTURBATION,
    'pair_selection_seed': PAIR_SELECTION_SEED,

    # General matrix construction.
    'batch_size': 4096,
    'matrix_batch_size': 4096,
    'sampling_replace': SAMPLING_REPLACE,
    'store_sampled_control_positions': True,

    # Random-average baseline.
    's2_n_control_cells_to_average_values': S2_N_CONTROL_CELLS_TO_AVERAGE_VALUES,

    # SEACell membership settings.
    'membership_root': str(OUTDIR / DATASET_ID / '_seacell_memberships'),
    'seacell_settings': SEACELL_SETTINGS,
    'seacell_key_prefix': 'SEACell',
    'n_waypoint_eigs': 10,
    'seacells_n_iter': 50,
    'seacell_seed': 42,
    'use_gpu_seacells': False,
    'overwrite_memberships': False,

    # S3 metacell-average baseline.
    's3_n_metacells_to_average_values': S3_N_METACELLS_TO_AVERAGE_VALUES,
    'sample_metacells_with_replacement': SAMPLE_METACELLS_WITH_REPLACEMENT,

    # S5 OT settings.
    's5_top_k_values': S5_TOP_K_VALUES,
    'sample_cells_per_metacell': SAMPLE_CELLS_PER_METACELL,
    'ot_reg': OT_REG,
    'ot_max_iter': 2000,
    'ot_tol': 1e-7,
    'cost_metric': 'sqeuclidean',
    'control_mass': CONTROL_MASS,
    'overwrite_assignments': False,
    'overwrite_sampled_outputs': False,
}

CONFIG


{'dataset_id': 'Replogle_RPE',
 'control_h5ad': '/ibex/user/chenj0i/Perturbation/data/processed_data/Replogle_RPE/groups/Replogle_RPE_control_processed.h5ad',
 'perturbed_h5ads': {'single': '/ibex/user/chenj0i/Perturbation/data/processed_data/Replogle_RPE/groups/Replogle_RPE_single_processed.h5ad'},
 'outdir': '/ibex/project/c2366/Perturb_data/Replogle_rpe_data',
 'strategies_to_run': ['S0_naive_mean_control_reference',
  'S1_random_single_control',
  'S2_random_average_controls',
  'S3_SEACell_metacell_average',
  'S4_SEACell_balanced_random_sample',
  'S5_SEACell_OT_sampled_average'],
 'expr_layer': 'X',
 'embedding_key': 'X_pca',
 'perturbation_key': 'auto',
 'require_all_genes': False,
 'random_seeds': [0, 1, 2, 3, 4],
 'max_pairs_per_perturbation': None,
 'pair_selection_seed': 42,
 'batch_size': 4096,
 'matrix_batch_size': 4096,
 'sampling_replace': True,
 'store_sampled_control_positions': True,
 's2_n_control_cells_to_average_values': [100],
 'membership_root': '/ibex/project/c

## 4. Optional path check

This cell does not load the h5ad matrices; it only checks whether the configured files exist.


In [6]:
missing = []
if not Path(CONFIG['control_h5ad']).exists():
    missing.append(CONFIG['control_h5ad'])
for group, path in CONFIG['perturbed_h5ads'].items():
    if not Path(path).exists():
        missing.append(path)

if missing:
    print('Missing files:')
    for p in missing:
        print(' -', p)
    raise FileNotFoundError('Fix the missing paths above before running the workflow.')
else:
    print('All configured h5ad files exist.')


All configured h5ad files exist.


## 5. Run pseudo-control construction

This will write one manifest at:

```text
{OUTDIR}/{DATASET_ID}/pseudo_pairing_repetition_manifest.csv
```

Each row points to one generated `pseudo_control_aligned_to_perturbed.h5ad` and its `pair_metadata` file.


In [7]:
RUN_PAIRING = True

if RUN_PAIRING:
    manifest = run_pseudo_pairing_repetition_plan(CONFIG)
    display(manifest)
else:
    print('RUN_PAIRING is False. Config prepared but workflow not launched.')



##############################################################################################################
[Perturbed group] single
[Perturbed h5ad]  /ibex/user/chenj0i/Perturbation/data/processed_data/Replogle_RPE/groups/Replogle_RPE_single_processed.h5ad
##############################################################################################################
[Membership] Reusing existing membership for nmc_50: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/_seacell_memberships/nmc_50/membership/control_cell_to_metacell_membership.csv
[Membership] Reusing existing membership for nmc_100: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/_seacell_memberships/nmc_100/membership/control_cell_to_metacell_membership.csv
[Membership] Reusing existing membership for nmc_200: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/_seacell_memberships/nmc_200/membership/control_cell_to_metacell_membership.csv
[Membership] Reusing existing membe

Aggregating groups by mean:   0%|          | 0/50 [00:00<?, ?it/s]

[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_50/k_03/seed_000/pseudo_control_aligned_to_perturbed.h5ad
[S3] Existing output found, skipping generation: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_50/k_03/seed_000/pseudo_control_aligned_to_perturbed.h5ad
[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_50/k_03/seed_001/pseudo_control_aligned_to_perturbed.h5ad
[S3] Existing output found, skipping generation: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_50/k_03/seed_001/pseudo_control_aligned_to_perturbed.h5ad
[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/c236

Aggregating groups by mean:   0%|          | 0/100 [00:00<?, ?it/s]

[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_100/k_03/seed_000/pseudo_control_aligned_to_perturbed.h5ad
[S3] Existing output found, skipping generation: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_100/k_03/seed_000/pseudo_control_aligned_to_perturbed.h5ad
[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_100/k_03/seed_001/pseudo_control_aligned_to_perturbed.h5ad
[S3] Existing output found, skipping generation: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_100/k_03/seed_001/pseudo_control_aligned_to_perturbed.h5ad
[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/

...

Aggregating groups by mean:   0%|          | 0/350 [00:00<?, ?it/s]

[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_350/k_03/seed_000/pseudo_control_aligned_to_perturbed.h5ad
[S3] Existing output found, skipping generation: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_350/k_03/seed_000/pseudo_control_aligned_to_perturbed.h5ad
[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_350/k_03/seed_001/pseudo_control_aligned_to_perturbed.h5ad
[S3] Existing output found, skipping generation: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_350/k_03/seed_001/pseudo_control_aligned_to_perturbed.h5ad
[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/

Aggregating groups by mean:   0%|          | 0/500 [00:00<?, ?it/s]

[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_500/k_03/seed_000/pseudo_control_aligned_to_perturbed.h5ad
[S3] Existing output found, skipping generation: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_500/k_03/seed_000/pseudo_control_aligned_to_perturbed.h5ad
[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_500/k_03/seed_001/pseudo_control_aligned_to_perturbed.h5ad
[S3] Existing output found, skipping generation: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/single/S3_SEACell_metacell_average/nmc_500/k_03/seed_001/pseudo_control_aligned_to_perturbed.h5ad
[S3_SEACell_metacell_average] Existing output found; skip generation and reuse: /ibex/project/

,dataset_id,perturbed_group,strategy,sampling_seed,parameter_label,pseudo_control_h5ad,pair_metadata_path,outdir,n_pairs_written,n_genes_output,...,n_metacells_observed,n_metacells_to_average,membership_path_for_metacell_coverage,n_unique_metacells_used,top_k_metacells,sample_cells_per_metacell,assignment_path,mean_dominant_weight,mean_matching_entropy,mean_dominant_distance
0,Replogle_RPE,single,S0_naive_mean_control_reference,NaN,global_control_mean,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Replogle_RPE,single,S1_random_single_control,0.0,seed_000,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Replogle_RPE,single,S1_random_single_control,1.0,seed_001,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Replogle_RPE,single,S1_random_single_control,2.0,seed_002,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Replogle_RPE,single,S1_random_single_control,3.0,seed_003,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,Replogle_RPE,single,S5_SEACell_OT_sampled_average,0.0,nmc_500_topk_5_seed_000,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,500.0,NaN,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,NaN,5.0,10.0,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,0.295977,0.622628,76.014367
182,Replogle_RPE,single,S5_SEACell_OT_sampled_average,1.0,nmc_500_topk_5_seed_001,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,500.0,NaN,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,NaN,5.0,10.0,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,0.295977,0.622628,76.014367
183,Replogle_RPE,single,S5_SEACell_OT_sampled_average,2.0,nmc_500_topk_5_seed_002,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,500.0,NaN,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,NaN,5.0,10.0,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,0.295977,0.622628,76.014367
184,Replogle_RPE,single,S5_SEACell_OT_sampled_average,3.0,nmc_500_topk_5_seed_003,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,500.0,NaN,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,NaN,5.0,10.0,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,0.295977,0.622628,76.014367


## 6. Inspect manifest summary


In [8]:
manifest_path = Path(CONFIG['outdir']) / CONFIG['dataset_id'] / 'pseudo_pairing_repetition_manifest.csv'
if manifest_path.exists():
    manifest = __import__('pandas').read_csv(manifest_path)
    print('Manifest:', manifest_path)
    print('Rows:', manifest.shape[0])
    display(
        manifest.groupby(['perturbed_group', 'strategy'], dropna=False)
        .agg(n_datasets=('pseudo_control_h5ad', 'count'))
        .reset_index()
    )
    display(manifest.head())
else:
    print('Manifest not found yet:', manifest_path)


Manifest: /ibex/project/c2366/Perturb_data/Replogle_rpe_data/Replogle_RPE/pseudo_pairing_repetition_manifest.csv
Rows: 186


,perturbed_group,strategy,n_datasets
0,single,S0_naive_mean_control_reference,1
1,single,S1_random_single_control,5
2,single,S2_random_average_controls,5
3,single,S3_SEACell_metacell_average,75
4,single,S4_SEACell_balanced_random_sample,25
5,single,S5_SEACell_OT_sampled_average,75


,dataset_id,perturbed_group,strategy,sampling_seed,parameter_label,pseudo_control_h5ad,pair_metadata_path,outdir,n_pairs_written,n_genes_output,...,n_metacells_observed,n_metacells_to_average,membership_path_for_metacell_coverage,n_unique_metacells_used,top_k_metacells,sample_cells_per_metacell,assignment_path,mean_dominant_weight,mean_matching_entropy,mean_dominant_distance
0,Replogle_RPE,single,S0_naive_mean_control_reference,NaN,global_control_mean,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Replogle_RPE,single,S1_random_single_control,0.0,seed_000,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Replogle_RPE,single,S1_random_single_control,1.0,seed_001,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Replogle_RPE,single,S1_random_single_control,2.0,seed_002,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Replogle_RPE,single,S1_random_single_control,3.0,seed_003,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,/ibex/project/c2366/Perturb_data/Replogle_rpe_...,236429,8749,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
